# Notebook 1 — Anatomy of a robust LLM call: what breaks in production, and how we absorb it

This notebook reads and runs the **LLM layer** of the testbench (`src/llm_testbench/llm/`, ADR 006).
It implements nothing: every block of code is imported from `src/` and run as is.

**What you will know by the end**: why an LLM call "that works" is not a robust call, what
actually breaks in production (rate limits, outages, timeouts, invalid JSON, costs, data going
where it should not), and how each link of our layer absorbs it — looking each time at **what
goes out on the network** and **what comes back**.

**How to read**: the configuration cells are at the top; the failure scenarios are played on a
**simulated network** (the real SDK, fake HTTP responses), so they run offline, in CI as on
your machine. The last part runs a real call if a key is available.

> Minimal vocabulary, defined as we go; summary glossary at the end of the notebook.

## Part 1 — The problem

An **LLM call** is an HTTP request to a remote service (the **provider**: Anthropic, OpenAI…)
that receives a **prompt** (text, split into **tokens**, the billing unit) and returns a
**completion** (generated text, also counted in tokens). Nothing more exotic than an API call —
except that the service is slow (seconds), expensive (billed per token), non-deterministic, and
that it goes down or throttles you more often than a database.

The naive call fits in five lines with the official SDK. And here is what happens to it in production:

| What breaks | Symptom | What you need |
|---|---|---|
| Rate limit (HTTP 429) | "too many requests", sometimes with a suggested delay | retry, after the suggested delay |
| Provider overloaded or down (529, 5xx, dropped connection) | bursts of errors | retry a little, then **stop insisting** (circuit breaker) |
| Response that never comes | the call hangs | a **timeout** beyond which it is an outage |
| Truncated response | text cut off in the middle | read the **finish reason** (`length`), not just the text |
| JSON requested, invalid JSON received | parsing blows up | validate, and **repair** by sending the error back to the model |
| Cost that drifts | surprise bill | a **budget** checked before the call, a cost computed after |
| The same question a thousand times | a thousand times the price | an exact **cache**, then a semantic one |
| Data sent to the wrong place | compliance incident | a **governance policy** checked before any call |

Each row of this table is a module of `src/llm_testbench/llm/`. We will read them, then knock
them down one by one.

In [1]:
# --- Configuration: edit, then "Run All" ---
import os

from llm_testbench.config import find_repo_root

# Sample mode (CI): simulated network only. LLM_TESTBENCH_SAMPLE=1 enables it.
SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"

# The notebook runs from notebooks/phase-1/: everything is anchored at the repository root.
ROOT = find_repo_root()

# Loads .env (API keys, at the repository root) into the environment, with no dependency.
ENV_FILE = ROOT / ".env"
if ENV_FILE.is_file():
    for line in ENV_FILE.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# The real call (part 4.9) only happens in full mode AND with a key.
USE_REAL_API = (not SAMPLE) and bool(os.environ.get("ANTHROPIC_API_KEY"))

# Model as "backend/model" (config/llm_backends.toml). Try "openai/gpt-5.6-luna".
MODEL = "anthropic/claude-sonnet-5-5"

# Knobs for the experiments (part 6):
RETRY_MAX_ATTEMPTS = 3  # maximum attempts per call
BREAKER_THRESHOLD = 2  # consecutive failures before the circuit opens
TIMEOUT_S = 0.2  # maximum duration of a call (the scenarios simulate slow responses)
SEMANTIC_THRESHOLD = 0.95  # minimum similarity for a semantic cache hit
DATA_CLASS = "public"  # try "regulated" to see governance refuse

print(f"sample mode: {SAMPLE} — real API call: {USE_REAL_API} — model: {MODEL}")

sample mode: True — real API call: False — model: anthropic/claude-sonnet-5-5


In [2]:
import inspect
import json
import random
import time

from IPython.display import Markdown, display
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from llm_testbench.llm import CompletionRequest, DataClass, Message, ModelRef, Role, Usage
from llm_testbench.llm.budget import TokenBudget
from llm_testbench.llm.cache import InMemoryCache, exact_key, semantic_namespace
from llm_testbench.llm.client import LLMClient, SemanticCacheConfig
from llm_testbench.llm.errors import (
    BudgetExceededError,
    GovernanceViolation,
    LLMError,
    ProviderTimeoutError,
)
from llm_testbench.llm.governance import GovernancePolicy
from llm_testbench.llm.prompts import PromptRegistry
from llm_testbench.llm.providers.anthropic import AnthropicProvider
from llm_testbench.llm.resilience import CircuitBreaker, RetryPolicy
from llm_testbench.llm.structured import StructuredOutputs
from llm_testbench.llm.testing import (
    ScriptedTransport,
    Step,
    ToyEmbedder,
    anthropic_error,
    anthropic_message,
    anthropic_sse,
    scripted_anthropic_provider,
)
from llm_testbench.obs.tracing import configure_tracing, get_tracer


def show_source(*objects: object) -> None:
    """Displays the real source of objects from src/ (display only, no logic here)."""
    display(Markdown("\n".join(f"```python\n{inspect.getsource(o)}```" for o in objects)))


def show_span(exporter: InMemorySpanExporter, prefixes=("gen_ai.", "llm_testbench.")) -> None:
    """Displays the attributes of the last exported span (what a trace backend would receive)."""
    span = exporter.get_finished_spans()[-1]
    attrs = dict(span.attributes or {})
    print(f'span "{span.name}" — status {span.status.status_code.name}')
    for key in sorted(k for k in attrs if k.startswith(prefixes)):
        print(f"  {key:<46} {attrs[key]!r}")
    exporter.clear()


def ask(
    question: str, *, system: str = "Answer in one sentence.", data_class: str = DATA_CLASS, **kw
) -> CompletionRequest:
    return CompletionRequest(
        model=ModelRef.parse(MODEL),
        messages=(
            Message(role=Role.SYSTEM, content=system),
            Message(role=Role.USER, content=question),
        ),
        data_class=DataClass(data_class),
        max_output_tokens=200,
        **kw,
    )


print(f"repository root: {ROOT}")

repository root: /Users/christophe/Roadmaps/roadmap-ai-engineer


### Toy example: the naive call, and what it does when the provider is overloaded

Before our layer, let us look at the SDK alone. We plug a **simulated network** into it: the request
really goes through the SDK (body construction, headers, serialization), but the HTTP response is
the one we decided on — here a **529 "overloaded"**, the error Anthropic returns at peak
times. The SDK retries on its own twice by default; we disable that (`max_retries=0`) to
see the raw error, and because our layer handles retries itself (part 3.4).

In [3]:
import anthropic

script = ScriptedTransport([anthropic_error(529, "Overloaded")])
naive_client = anthropic.AsyncAnthropic(
    api_key="fake-key",
    max_retries=0,
    http_client=anthropic.DefaultAsyncHttpxClient(transport=script.transport()),
)
try:
    await naive_client.messages.create(
        model="claude-sonnet-5",
        max_tokens=100,
        messages=[{"role": "user", "content": "What is the capital of France?"}],
    )
except Exception as error:
    print(f"raw SDK exception: {type(error).__module__}.{type(error).__name__}")
    print(f"message: {error}")

print("\nbody actually sent by the SDK:")
print(json.dumps(script.requests[0], indent=2, ensure_ascii=False))

raw SDK exception: anthropic.OverloadedError
message: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}}

body actually sent by the SDK:
{
  "max_tokens": 100,
  "messages": [
    {
      "role": "user",
      "content": "What is the capital of France?"
    }
  ],
  "model": "claude-sonnet-5"
}


The SDK did its job: it raised a typed exception. But the naive call stops there:
no retry, no timeout, no failure counter, no trace, no cost.
Everything that follows consists of *wrapping* this SDK, without replacing it.

## Part 2 — The options

| Option | What it brings | Why not (ADR 006) |
|---|---|---|
| **LiteLLM** or equivalent | a hundred providers behind a single API | hides precisely the mechanics we want to learn; heavy dependency |
| **OpenRouter** as the only path | a single OpenAI-compatible client, huge catalog | one more intermediary for data governance, no batch API, normalized token counting |
| **Our layer on the official SDKs** ✔ | two implementations (native Anthropic, OpenAI-compatible) for six backends; each link readable and tested | more code of our own — that is the point |

OpenRouter remains a *breadth backend* (Mistral, Qwen, Llama without a dedicated key), never the
only path; Bedrock (EU residency) goes through the Bedrock client of the Anthropic SDK; vLLM
(phase 6) through the OpenAI-compatible client.

## Part 3 — Guided reading of the real code

### 3.1 The neutral request: SDK-independent, frozen, and governed

Everything starts with `CompletionRequest`. Two choices to notice: it is **frozen** (`frozen`) — we
do not modify it, we derive from it — and its **data class** (`data_class`) is **mandatory**,
with no default value: the implicit default would be the first governance hole.

In [4]:
show_source(CompletionRequest, Usage)
request = ask("What is the capital of France?")
print(json.dumps(request.model_dump(mode="json"), indent=2, ensure_ascii=False))

```python
class CompletionRequest(BaseModel):
    model_config = ConfigDict(frozen=True)

    model: ModelRef
    messages: tuple[Message, ...] = Field(min_length=1)
    data_class: DataClass
    """Required, with no default: the implicit default would be the first hole."""

    max_output_tokens: int = Field(default=1024, ge=1)
    temperature: float | None = Field(default=None, ge=0.0, le=2.0)
    """``None``: provider default. ``0`` does not guarantee determinism (phase 5B)."""

    stop: tuple[str, ...] = ()
    seed: int | None = None
    json_schema: dict[str, Any] | None = None
    """JSON schema expected as output. The provider enforces it natively if it can, otherwise
    the facade injects it into the prompt and validates afterwards (``structured.py``)."""

    reasoning_effort: Literal["low", "medium", "high"] | None = None
    """Reasoning effort, when the backend has this setting (``output_config.effort`` at
    Anthropic, ``reasoning_effort`` at OpenAI). ``None``: provider default."""

    timeout_s: float | None = Field(default=None, gt=0)
    metadata: dict[str, str] = Field(default_factory=dict)
    """Trace information (prompt name/version/fingerprint…). Not part of the cache key."""

    @field_validator("messages")
    @classmethod
    def _system_only_first(cls, messages: tuple[Message, ...]) -> tuple[Message, ...]:
        for index, message in enumerate(messages):
            if message.role is Role.SYSTEM and index != 0:
                raise ValueError("a system message is only allowed in first position")
        return messages

    @property
    def system(self) -> str | None:
        first = self.messages[0]
        return first.content if first.role is Role.SYSTEM else None

    @property
    def turns(self) -> tuple[Message, ...]:
        """The non-system messages, in order."""
        return tuple(m for m in self.messages if m.role is not Role.SYSTEM)

    def with_messages(self, messages: tuple[Message, ...]) -> "CompletionRequest":
        return self.model_copy(update={"messages": messages})
```
```python
class Usage(BaseModel):
    """Tokens consumed, normalized across providers.

    ``input_tokens`` counts the input tokens **not served by the cache** (billed at full
    price). Anthropic reports them this way natively; OpenAI includes cached tokens in
    ``prompt_tokens``, and the provider does the subtraction. Without this normalization, the
    per-request cost of phase 5 would compare different things.
    """

    model_config = ConfigDict(frozen=True)

    input_tokens: int = Field(default=0, ge=0)
    output_tokens: int = Field(default=0, ge=0)
    cache_read_tokens: int = Field(default=0, ge=0)
    cache_write_tokens: int = Field(default=0, ge=0)

    @property
    def total_input_tokens(self) -> int:
        return self.input_tokens + self.cache_read_tokens + self.cache_write_tokens

    @property
    def total_tokens(self) -> int:
        return self.total_input_tokens + self.output_tokens

    def __add__(self, other: "Usage") -> "Usage":
        return Usage(
            input_tokens=self.input_tokens + other.input_tokens,
            output_tokens=self.output_tokens + other.output_tokens,
            cache_read_tokens=self.cache_read_tokens + other.cache_read_tokens,
            cache_write_tokens=self.cache_write_tokens + other.cache_write_tokens,
        )
```

{
  "model": {
    "backend": "anthropic",
    "model": "claude-sonnet-5-5"
  },
  "messages": [
    {
      "role": "system",
      "content": "Answer in one sentence."
    },
    {
      "role": "user",
      "content": "What is the capital of France?"
    }
  ],
  "data_class": "public",
  "max_output_tokens": 200,
  "temperature": null,
  "stop": [],
  "seed": null,
  "json_schema": null,
  "reasoning_effort": null,
  "timeout_s": null,
  "metadata": {}
}


### 3.2 Translation to the SDK, and typed errors

The **provider** translates the neutral request into SDK parameters. `build_params` is public
on purpose: we can look at what is going to go out *before* it goes out. Two translations are surprising:

- `temperature` and `seed` **disappear** — current Claude models (Sonnet 5, Opus 5) reject
  any sampling parameter with a 400. The "temperature 0 for determinism" reflex
  is a 2024 reflex;
- the system prompt is marked **cacheable** (`cache_control`): Anthropic only caches what
  it is asked to cache, and only above a minimum size (part 4.9).

Then `translate_error` converts the SDK exceptions into **typed errors of our layer**, with
a `retryable` flag: it is that flag, and that flag alone, that the retry will consult.

In [5]:
show_source(AnthropicProvider.build_params, AnthropicProvider.translate_error)

provider, script = scripted_anthropic_provider([Step(json_body=anthropic_message())])
params = provider.build_params(ask("Hello", temperature=0.0, seed=42, reasoning_effort="low"))
print("SDK parameters built for the request (temperature and seed absent, effort present):")
print(json.dumps(params, indent=2, ensure_ascii=False))

```python
    def build_params(self, request: CompletionRequest) -> dict[str, Any]:
        """The neutral request → the SDK parameters. Public: the notebook displays it."""
        params: dict[str, Any] = {
            "model": request.model.model,
            "max_tokens": request.max_output_tokens,
            "messages": [{"role": m.role.value, "content": m.content} for m in request.turns],
        }
        if request.system is not None:
            block: dict[str, Any] = {"type": "text", "text": request.system}
            if self.cache_system_prompt:
                block["cache_control"] = {"type": "ephemeral"}
            params["system"] = [block]
        if request.stop:
            params["stop_sequences"] = list(request.stop)
        output_config: dict[str, Any] = {}
        if request.reasoning_effort is not None:
            output_config["effort"] = request.reasoning_effort
        if request.json_schema is not None and self.native_json_schema:
            output_config["format"] = {"type": "json_schema", "schema": request.json_schema}
        if output_config:
            params["output_config"] = output_config
        if request.timeout_s is not None:
            params["timeout"] = request.timeout_s
        return params
```
```python
    def translate_error(self, error: Exception) -> LLMError:
        backend = self.backend
        if isinstance(error, anthropic.RateLimitError):
            return RateLimitedError(
                str(error),
                backend=backend,
                retry_after_s=retry_after_seconds(error.response.headers),
            )
        if isinstance(error, anthropic.APITimeoutError):
            return ProviderTimeoutError(str(error), backend=backend)
        if isinstance(error, anthropic.APIConnectionError):
            return ProviderUnavailableError(str(error), backend=backend)
        if isinstance(error, anthropic.APIStatusError):
            if error.status_code >= 500:
                return ProviderUnavailableError(
                    str(error), backend=backend, status_code=error.status_code
                )
            return InvalidRequestError(str(error), backend=backend, status_code=error.status_code)
        return ProviderUnavailableError(f"unexpected error: {error!r}", backend=backend)
```

SDK parameters built for the request (temperature and seed absent, effort present):
{
  "model": "claude-sonnet-5-5",
  "max_tokens": 200,
  "messages": [
    {
      "role": "user",
      "content": "Hello"
    }
  ],
  "system": [
    {
      "type": "text",
      "text": "Answer in one sentence.",
      "cache_control": {
        "type": "ephemeral"
      }
    }
  ],
  "output_config": {
    "effort": "low"
  }
}


### 3.3 Retry, circuit breaker, timeout: three different questions

- **Retry** — "this attempt failed, can the next one succeed?" Yes for a 429, a
  5xx, a dropped connection; no for a 400. The delay between attempts grows (**exponential backoff**)
  with a random **jitter**: without it, all clients retry at the same second and
  recreate the outage. A `Retry-After` from the provider is honored, capped.
- **Circuit breaker** — "is this backend going down?" After N consecutive failures,
  we stop calling it for a while (**open**), then let a trial call through
  (**half-open**): success → closed, failure → reopened. We protect the provider *and* our callers,
  who fail fast instead of waiting.
- **Timeout** — a silent provider is an outage, not a wait.

Time and randomness are **injected**: the tests play hours in milliseconds, and this
notebook never actually waits.

In [6]:
show_source(RetryPolicy.delay_for, CircuitBreaker.before_call, CircuitBreaker.record_failure)

policy = RetryPolicy(max_attempts=5, base_delay_s=0.5, max_delay_s=8.0, jitter=0.25)
rng = random.Random(7)
print("attempt   | delay before the next one (backoff × jitter)")
for attempt in range(1, 6):
    print(f"    {attempt}     | {policy.delay_for(attempt, None, rng):6.2f} s")
print(
    f"with Retry-After: 3 s → {policy.delay_for(1, 3.0, rng):.1f} s; Retry-After: 60 s → capped at {policy.delay_for(1, 60.0, rng):.1f} s"
)

```python
    def delay_for(self, attempt: int, retry_after_s: float | None, rng: random.Random) -> float:
        """Delay before attempt ``attempt + 1`` (``attempt`` counts from 1).

        A ``Retry-After`` from the provider is honored, capped at ``max_delay_s``: beyond that,
        we prefer to retry early and fail fast rather than block a caller.
        """
        if retry_after_s is not None:
            return min(max(retry_after_s, 0.0), self.max_delay_s)
        raw = self.base_delay_s * self.multiplier ** (attempt - 1)
        capped = min(raw, self.max_delay_s)
        return capped * (1.0 + rng.uniform(-self.jitter, self.jitter))
```
```python
    def before_call(self) -> None:
        state = self.state
        if state is CircuitState.OPEN:
            assert self._opened_at is not None
            remaining = self.recovery_timeout_s - (self._clock() - self._opened_at)
            raise CircuitOpenError(backend=self.backend, retry_in_s=max(remaining, 0.0))
        if state is CircuitState.HALF_OPEN:
            if self._trial_in_flight:
                raise CircuitOpenError(backend=self.backend, retry_in_s=0.0)
            self._trial_in_flight = True
```
```python
    def record_failure(self) -> None:
        self._consecutive_failures += 1
        if self._state is CircuitState.HALF_OPEN or (
            self._consecutive_failures >= self.failure_threshold
        ):
            self._state = CircuitState.OPEN
            self._opened_at = self._clock()
            self._trial_in_flight = False
```

attempt   | delay before the next one (backoff × jitter)
    1     |   0.46 s
    2     |   0.83 s
    3     |   2.15 s
    4     |   3.14 s
    5     |   8.14 s
with Retry-After: 3 s → 3.0 s; Retry-After: 60 s → capped at 8.0 s


### 3.4 Budget, cache, governance: refuse or avoid *before* calling

- The **budget** refuses a call based on an estimate (four characters per token, pessimistic) and
  reserves the maximum output: we refuse what *could* exceed it.
- The **exact cache** is keyed on the fingerprint of everything that influences the response — and of
  nothing else (trace metadata does not go into it). The **semantic cache** compares the embedding
  of the user message within a **namespace** that freezes everything else (model, system prompt,
  parameters): without it, "summarize this text" served for another system prompt would return the
  wrong answer with perfect similarity.
- The **governance policy** maps a data class to requirements on the backend
  (inference region, zero retention, training, subprocessors). Unknown metadata
  fails: the unknown is never a free pass.

In [7]:
show_source(exact_key, semantic_namespace, GovernancePolicy.evaluate)

policy_cfg = GovernancePolicy.load(ROOT / "config" / "governance_policy.toml")
print("policy rules (config/governance_policy.toml):")
for data_class, rule in policy_cfg.rules.items():
    print(
        f"  {data_class.value:<13} {rule.model_dump(exclude_defaults=True) or '(no requirement)'}"
    )

```python
def exact_key(request: CompletionRequest) -> str:
    return _digest(
        {
            "model": str(request.model),
            "messages": [(m.role.value, m.content) for m in request.messages],
            **_params(request),
        }
    )
```
```python
def semantic_namespace(request: CompletionRequest) -> str | None:
    """Namespace of the semantic cache, or ``None`` if the request is not eligible
    (more than one conversation turn)."""
    turns = request.turns
    if len(turns) != 1 or turns[0].role is not Role.USER:
        return None
    return _digest({"model": str(request.model), "system": request.system, **_params(request)})
```
```python
    def evaluate(
        self, data_class: DataClass, backend: str, governance: BackendGovernance
    ) -> GovernanceDecision:
        rule = self.rules[data_class]
        reasons: list[str] = []

        region = governance.inference_region
        if rule.allowed_regions is not None and region not in rule.allowed_regions:
            allowed = ", ".join(r.value for r in rule.allowed_regions)
            reasons.append(f"inference region {region.value!r} outside [{allowed}]")
        if rule.require_zero_retention and governance.zero_retention is not True:
            state = "unknown" if governance.zero_retention is None else "not configured"
            reasons.append(f"zero retention required, {state}")
        if rule.forbid_training and governance.trains_on_inputs is not False:
            state = "unknown" if governance.trains_on_inputs is None else "declared"
            reasons.append(f"training on inputs forbidden, {state}")
        if rule.forbid_variable_subprocessors and governance.subprocessors != "fixed":
            reasons.append("variable subprocessors (broker) forbidden")

        return GovernanceDecision(
            data_class=data_class, backend=backend, allowed=not reasons, reasons=tuple(reasons)
        )
```

policy rules (config/governance_policy.toml):
  public        (no requirement)
  internal      {'forbid_training': True}
  confidential  {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}
  personal      {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}
  regulated     {'allowed_regions': (<InferenceRegion.EU: 'eu'>, <InferenceRegion.SELF: 'self'>), 'require_zero_retention': True, 'forbid_training': True, 'forbid_variable_subprocessors': True}


### 3.5 The facade: the order of the links is the contract

`LLMClient.complete` chains, in this order: governance → budget → cache → breaker → retry →
provider (with timeout) → budget → cost → trace. Read it in full once: everything the
scenarios of part 4 will show is in these eighty lines.

In [8]:
show_source(LLMClient.complete)

```python
    async def complete(self, request: CompletionRequest, *, use_cache: bool = True) -> Completion:
        with self._tracer.start_as_current_span(f"chat {request.model}") as span:
            self._request_attributes(span, request, "chat")
            ref, config = self.registry.resolve(request.model)
            backend = ref.backend
            self._govern(span, request.data_class, backend, config)

            estimated = sum(estimate_tokens(m.content) for m in request.messages)
            if self.budget is not None:
                self.budget.check(
                    estimated_input_tokens=estimated, max_output_tokens=request.max_output_tokens
                )

            key = exact_key(request)
            namespace = semantic_namespace(request)
            vector: tuple[float, ...] | None = None
            if self.cache is not None and use_cache:
                hit, vector = await self._cache_lookup(span, request, key, namespace)
                if hit is not None:
                    cached = hit.model_copy(
                        update={"cached": True, "cost_usd": 0.0, "latency_ms": 0.0}
                    )
                    span.set_attribute(Bench.CACHED, True)
                    self._record_usage(span, Usage(), 0.0)
                    return cached
            span.set_attribute(Bench.CACHED, False)

            provider = self.provider_for(backend, config)
            completion, report = await self._guarded(
                backend, request.timeout_s, lambda: provider.complete(request)
            )

            if self.budget is not None:
                self.budget.record(completion.usage)
            cost = self._cost(completion.model, completion.usage, completion.usage_reported)
            completion = completion.model_copy(update={"cost_usd": cost, "attempts": report.count})

            span.set_attribute(GenAI.RESPONSE_MODEL, completion.model)
            span.set_attribute(GenAI.RESPONSE_FINISH_REASONS, [completion.finish_reason.value])
            if completion.provider_message_id:
                span.set_attribute(GenAI.RESPONSE_ID, completion.provider_message_id)
            span.set_attribute(Bench.LATENCY_MS, completion.latency_ms)
            span.set_attribute(Bench.ATTEMPTS, report.count)
            self._record_usage(span, completion.usage, cost)

            if self.cache is not None and use_cache:
                await self.cache.put(key, completion)
                if namespace is not None and vector is not None:
                    await self.cache.put_vector(namespace, vector, key)
            return completion
```

## Part 4 — Run: knock down each link, and watch

The setup is the same for all scenarios: an `LLMClient` built on the project's **real
configuration** (`config/`), an **in-memory** trace exporter to read the spans,
a `sleep` that **records** the delays instead of sleeping, and a real Anthropic provider plugged into
the scripted network. The `scenario` function below is just notebook glue.

In [9]:
class FakeClock:
    """Clock injected into the breaker: we move time forward by hand."""

    def __init__(self) -> None:
        self.now = 0.0

    def __call__(self) -> float:
        return self.now


def scenario(
    steps,
    *,
    attempts=None,
    breaker_threshold=None,
    recovery_s=30.0,
    clock=None,
    timeout_s=None,
    **extra,
):
    exporter = InMemorySpanExporter()
    tracer = get_tracer(configure_tracing(exporter=exporter, set_global=False))
    sleeps: list[float] = []

    async def recording_sleep(delay: float) -> None:
        sleeps.append(delay)

    client = LLMClient.from_settings(
        tracer=tracer,
        retry_policy=RetryPolicy(
            max_attempts=attempts or RETRY_MAX_ATTEMPTS, base_delay_s=0.5, jitter=0.0
        ),
        sleep=recording_sleep,
        breaker_factory=lambda backend: CircuitBreaker(
            backend=backend,
            failure_threshold=breaker_threshold or BREAKER_THRESHOLD,
            recovery_timeout_s=recovery_s,
            clock=clock or time.monotonic,
        ),
        default_timeout_s=timeout_s or TIMEOUT_S,
        **extra,
    )
    provider, script = scripted_anthropic_provider(steps, backend=ModelRef.parse(MODEL).backend)
    client.register_provider(ModelRef.parse(MODEL).backend, provider)
    return client, script, exporter, sleeps


def describe(completion) -> None:
    print(f"text:         {completion.text!r}")
    print(f"served model: {completion.model}  (requested: {ModelRef.parse(MODEL).model})")
    print(f"usage:        {completion.usage.model_dump()}")
    print(
        f"cost:         {completion.cost_usd if completion.cost_usd is None else f'{completion.cost_usd:.6f} $'}"
    )
    print(
        f"attempts:     {completion.attempts} — cache: {completion.cached} — finish: {completion.finish_reason.value}"
    )


print("setup ready")

setup ready


### 4.1 Nominal path: one call, its cost, its trace

Look at the trace: the `gen_ai.*` attributes are OpenTelemetry's **GenAI semantic conventions**
(the standard vocabulary that Langfuse, Tempo or Jaeger can read), the
`llm_testbench.*` ones are what the testbench adds — cost, cache, attempts, governance, prompt fingerprint.

In [10]:
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris.", input_tokens=1_000, output_tokens=12))]
)
completion = await client.complete(ask("What is the capital of France?"))
describe(completion)
print()
show_span(exporter)

text:         'Paris.'
served model: claude-sonnet-5  (requested: claude-sonnet-5-5)
usage:        {'input_tokens': 1000, 'output_tokens': 12, 'cache_read_tokens': 0, 'cache_write_tokens': 0}
cost:         0.002120 $
attempts:     1 — cache: False — finish: stop

span "chat anthropic/claude-sonnet-5-5" — status UNSET
  gen_ai.operation.name                          'chat'
  gen_ai.provider.name                           'anthropic'
  gen_ai.request.max_tokens                      200
  gen_ai.request.model                           'claude-sonnet-5-5'
  gen_ai.response.finish_reasons                 ('stop',)
  gen_ai.response.id                             'msg_scripted'
  gen_ai.response.model                          'claude-sonnet-5'
  gen_ai.usage.input_tokens                      1000
  gen_ai.usage.output_tokens                     12
  llm_testbench.attempts                         1
  llm_testbench.backend                          'anthropic'
  llm_testbench.cache.hit         

### 4.2 Rate limit: 429 with `Retry-After`, then success

The provider says "too fast, come back in 2 s". Our retry honors the suggested delay (instead
of its own backoff), then retries. Two attempts, a single sleep, a single billing.

In [11]:
client, script, exporter, sleeps = scenario(
    [
        anthropic_error(429, "Too many requests", retry_after_s=2),
        Step(json_body=anthropic_message("Paris.")),
    ]
)
completion = await client.complete(ask("Capital of France?"))
describe(completion)
print(f"\nrequests sent over the network: {script.calls} — delays slept (simulated): {sleeps}")
show_span(exporter, prefixes=("llm_testbench.attempts",))

text:         'Paris.'
served model: claude-sonnet-5  (requested: claude-sonnet-5-5)
usage:        {'input_tokens': 120, 'output_tokens': 8, 'cache_read_tokens': 0, 'cache_write_tokens': 0}
cost:         0.000320 $
attempts:     2 — cache: False — finish: stop

requests sent over the network: 2 — delays slept (simulated): [2.0]
span "chat anthropic/claude-sonnet-5-5" — status UNSET
  llm_testbench.attempts                         2


### 4.3 Prolonged outage: the retry insists, then the breaker cuts off

The provider returns 529 in a loop. With `BREAKER_THRESHOLD` consecutive failures, the circuit
**opens**: the next attempt fails *without a network call*, and so does any new call, until
the end of the recovery timeout. We then move the clock forward by hand to see the circuit go
**half-open** and then close again on a success.

In [12]:
clock = FakeClock()
client, script, exporter, sleeps = scenario(
    [anthropic_error(529, "Overloaded")], clock=clock, recovery_s=30.0
)

for attempt in (1, 2):
    try:
        await client.complete(ask("Capital?"))
    except LLMError as error:
        print(f"call {attempt} → {type(error).__name__}: {error}")
    print(
        f"   cumulative network requests: {script.calls} — breaker: {client.breaker_for('anthropic').snapshot()}"
    )

clock.now += 31  # the recovery timeout has elapsed
client.register_provider(
    "anthropic", scripted_anthropic_provider([Step(json_body=anthropic_message("Paris."))])[0]
)
completion = await client.complete(ask("Capital?"))
print(
    f"\nafter 31 simulated s → trial call succeeded: {completion.text!r} — breaker: {client.breaker_for('anthropic').snapshot()}"
)

call 1 → CircuitOpenError: circuit open for backend 'anthropic', retry possible in 30.0 s
   cumulative network requests: 2 — breaker: {'backend': 'anthropic', 'state': 'open', 'consecutive_failures': 2}
call 2 → CircuitOpenError: circuit open for backend 'anthropic', retry possible in 30.0 s
   cumulative network requests: 2 — breaker: {'backend': 'anthropic', 'state': 'open', 'consecutive_failures': 2}

after 31 simulated s → trial call succeeded: 'Paris.' — breaker: {'backend': 'anthropic', 'state': 'closed', 'consecutive_failures': 0}


### 4.4 The provider does not answer: the timeout turns waiting into an outage

The response arrives… in 0.5 s, for an allowed timeout of `TIMEOUT_S`. Without a timeout, the caller
would wait indefinitely; with one, it is a `ProviderTimeoutError`, **retryable** (the provider may
have received nothing). Here we limit to one attempt to see the bare error.

In [13]:
client, script, exporter, sleeps = scenario(
    [Step(delay_s=0.5, json_body=anthropic_message("Paris."))], attempts=1
)
started = time.perf_counter()
try:
    await client.complete(ask("Capital?"))
except ProviderTimeoutError as error:
    print(
        f"{type(error).__name__} after {time.perf_counter() - started:.2f} s: {error} — retryable: {error.retryable}"
    )

ProviderTimeoutError after 0.20 s: no response from 'anthropic' in 0.2 s — retryable: True


### 4.5 Refusing before calling: governance and budget

A request labeled `regulated` cannot go to a US backend without zero
retention (provisional policy, `config/governance_policy.toml`). The refusal is a typed exception
**before any network call** (count the requests: zero), and the trace keeps the decision and its
reasons. Same logic for the budget: the estimate refuses, the provider is never called.

In [14]:
client, script, exporter, sleeps = scenario([Step(json_body=anthropic_message("Paris."))])
try:
    await client.complete(ask("Patient record no. 42?", data_class="regulated"))
except GovernanceViolation as error:
    print(f"GovernanceViolation: {error}")
print(f"network requests: {script.calls}")
show_span(exporter, prefixes=("llm_testbench.governance",))

tiny_budget = TokenBudget(max_input_tokens_per_request=10)
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris."))], budget=tiny_budget
)
try:
    await client.complete(ask("A question far too long for a budget of ten tokens?"))
except BudgetExceededError as error:
    print(f"\nBudgetExceededError: {error} — network requests: {script.calls}")

GovernanceViolation: backend 'anthropic' refused for data class 'regulated': inference region 'us' outside [eu, self]; zero retention required, not configured
network requests: 0
span "chat anthropic/claude-sonnet-5-5" — status ERROR
  llm_testbench.governance.allowed               False
  llm_testbench.governance.data_class            'regulated'
  llm_testbench.governance.inference_region      'us'
  llm_testbench.governance.reasons               ("inference region 'us' outside [eu, self]", 'zero retention required, not configured')

BudgetExceededError: input estimated at 19 tokens, per-request cap 10 — network requests: 0


### 4.6 The caches: exact, then semantic

Three calls. The second is an exact copy of the first: served from the exact cache, zero cost,
no network request. The third rephrases the question: the exact cache misses, the
**semantic** cache compares the embeddings (here a three-dimensional toy embedder, imported from
`src/`) and serves the response if the similarity exceeds `SEMANTIC_THRESHOLD`. Watch the network
request counter and the `cache.kind` attribute of the traces.

In [15]:
client, script, exporter, sleeps = scenario(
    [Step(json_body=anthropic_message("Paris.", input_tokens=800, output_tokens=6))],
    cache=InMemoryCache(),
    semantic_cache=SemanticCacheConfig(
        embedding_model=ModelRef.parse("openai/text-embedding-3-small"),
        min_similarity=SEMANTIC_THRESHOLD,
    ),
)
client.register_embedder("openai", ToyEmbedder())

for question in (
    "What is the capital of France?",
    "What is the capital of France?",
    "Capital of France, please?",
):
    completion = await client.complete(ask(question))
    attrs = dict(exporter.get_finished_spans()[-1].attributes or {})
    print(
        f"{question:<42} cache={completion.cached!s:<5} kind={attrs.get('llm_testbench.cache.kind', '-'):<8} cost={completion.cost_usd:.6f} $  network={script.calls}"
    )
    exporter.clear()

What is the capital of France?             cache=False kind=-        cost=0.001660 $  network=1
What is the capital of France?             cache=True  kind=exact    cost=0.000000 $  network=1
Capital of France, please?                 cache=True  kind=semantic cost=0.000000 $  network=1


### 4.7 Structured output: request a schema, validate, repair

We want a Pydantic object, not text. The Anthropic backend can **enforce** a JSON schema
(`output_config.format`); we see it in the parameters sent. But a JSON schema cannot
express everything (here, a confidence between 0 and 1): Pydantic validation remains, and when it
fails, the layer **sends the error back to the model** in a repair message — at most
`max_repairs` times. Look at the exact repair message: it is the versioned prompt
`prompts/structured_output_repair.md`, rendered.

In [16]:
from pydantic import BaseModel, Field


class Capital(BaseModel):
    country: str
    city: str
    confidence: float = Field(ge=0.0, le=1.0)


client, script, exporter, sleeps = scenario(
    [
        Step(
            json_body=anthropic_message('{"country": "France", "city": "Paris", "confidence": 7}')
        ),
        Step(
            json_body=anthropic_message(
                '{"country": "France", "city": "Paris", "confidence": 0.98}'
            )
        ),
    ]
)
structured = StructuredOutputs(client, PromptRegistry.from_settings(), max_repairs=2)
result = await structured.complete(ask("Capital of France, with your confidence?"), Capital)

print(f"validated value: {result.value!r} — attempts: {result.attempts}")
print(
    f"\nschema enforced on the provider (output_config): {json.dumps(script.requests[0]['output_config'], ensure_ascii=False)[:160]}…"
)
print(f"\nvalidation error sent back to the model:\n{result.repairs[0][:300]}")
print("\nrepair message actually sent (last message of the 2nd request):")
print(script.requests[1]["messages"][-1]["content"])

validated value: Capital(country='France', city='Paris', confidence=0.98) — attempts: 2

schema enforced on the provider (output_config): {"format": {"type": "json_schema", "schema": {"properties": {"country": {"title": "Country", "type": "string"}, "city": {"title": "City", "type": "string"}, "co…

validation error sent back to the model:
1 validation error for Capital
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

repair message actually sent (last message of the 2nd request):
Your previous answer does not conform to the expected schema. Validation error:

1 validation error for Capital
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=7, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

Return only the corrected JSON object, complete, with no surround

### 4.8 Streaming: tokens arrive as they come, usage at the end

In **streaming** (SSE), the provider sends the response chunk by chunk: the user sees the
first word after a few hundred milliseconds (the **TTFT**, time to first token) instead of
waiting for the whole sentence. The price: you cannot **retry** a stream already under way without
duplicating text already displayed — so the facade does not retry after the first chunk — and the usage
is only known at the end. The cost is computed on the last chunk.

In [17]:
client, script, exporter, sleeps = scenario(
    [Step(sse_body=anthropic_sse(["The ", "capital ", "is ", "Paris."]))]
)
started = time.perf_counter()
async for chunk in client.stream(ask("Capital?")):
    elapsed = (time.perf_counter() - started) * 1000
    if chunk.kind == "text":
        print(f"  +{elapsed:5.1f} ms  text   {chunk.text!r}")
    elif chunk.kind == "usage":
        print(
            f"  +{elapsed:5.1f} ms  usage  {chunk.usage.model_dump()}  cost={chunk.cost_usd:.6f} $"
        )
    else:
        print(f"  +{elapsed:5.1f} ms  end    {chunk.finish_reason.value}")

  + 15.4 ms  text   'The '
  + 15.6 ms  text   'capital '
  + 15.6 ms  text   'is '
  + 15.6 ms  text   'Paris.'
  + 18.8 ms  usage  {'input_tokens': 120, 'output_tokens': 8, 'cache_read_tokens': 0, 'cache_write_tokens': 0}  cost=0.000320 $
  + 18.8 ms  end    stop


### 4.9 With a real provider (full mode): prompt caching in numbers

This cell only runs in full mode with `ANTHROPIC_API_KEY`. It sends the same call twice
with a **long system prompt** (above the model's minimum cacheable size): on the first
call, the system prompt tokens are **written** to the cache (billed 1.25×); on the second,
**read** (0.1×). Compare `cache_write_tokens`, `cache_read_tokens` and the cost of the two calls:
it is the difference between a RAG that costs and a RAG that costs ten times less on its fixed prefix.

In [18]:
if not USE_REAL_API:
    print("section skipped: sample mode or ANTHROPIC_API_KEY missing (see .env.example)")
else:
    exporter = InMemorySpanExporter()
    real = LLMClient.from_settings(
        tracer=get_tracer(configure_tracing(exporter=exporter, set_global=False))
    )
    paragraph = (
        "You are the assistant of a testbench for LLM techniques. You answer in English, briefly, "
        "and you cite your sources when you are given some. This paragraph is repeated to exceed the "
        "minimum size for caching the prompt prefix. "
    )
    long_system = paragraph * 60  # ≈ 2,500 tokens
    for label in ("1st call", "2nd call"):
        completion = await real.complete(
            ask(
                "Answer simply: what is the capital of France?",
                system=long_system,
                max_output_tokens=30,
            ),
            use_cache=False,
        )
        u = completion.usage
        print(
            f"{label}: served model={completion.model} input={u.input_tokens} cache_write={u.cache_write_tokens} cache_read={u.cache_read_tokens} output={u.output_tokens} cost={completion.cost_usd:.5f} $ latency={completion.latency_ms:.0f} ms"
        )

section skipped: sample mode or ANTHROPIC_API_KEY missing (see .env.example)


## Part 5 — Pitfalls (what the tutorials do not say)

1. **Two layers of retry**: the SDK already retries 429 and 5xx (twice by default). Stacking
   our retry on top multiplies the attempts without anyone noticing. We set the SDK to
   zero: a single policy, ours, observable.
2. **`temperature=0` returns a 400** on current Claude models. Determinism is *measured*
   (phase 5B), it is no longer decreed by a parameter.
3. **`Retry-After` can say 60 s.** Honoring it to the letter blocks a caller for a minute; we
   cap it, and prefer to fail fast.
4. **The semantic cache lies with confidence**: two questions that are close in embedding terms
   can expect different answers ("capital of France" / "capital of
   Finland" depending on the embedding model). Hence a high threshold, a strict namespace, and
   a measurement of the false-positive rate before enabling it in production (phase 5).
5. **An unknown cost is not a zero cost.** A model missing from the price table yields
   `None`, and the trace says so (`cost_known=false`). A silent zero would propagate all the way
   to the master table.
6. **The served model is not necessarily the requested model**: the response carries the exact
   identifier, we trace it. A provider that changes revision without notice is the subject of a
   phase 5 runbook.
7. **Prompt caching has a minimum size** (on the order of a thousand tokens depending on the model):
   on a short prompt, `cache_read_tokens` stays at zero, and that is not a bug.
8. **A stream already under way is not retried.** If the provider goes down in the middle, the user
   has already seen half of the response: it is up to the caller to decide (start over, display the
   error), not up to the layer to duplicate text.
9. **Governance metadata are claims**, dated, collected from public
   pages: they are verified contractually, they are not assumed.

## Part 6 — Your turn

Edit the configuration at the top, then "Run All":

1. **`RETRY_MAX_ATTEMPTS = 1`** — scenario 4.2 fails on the first 429; at `5`, scenario 4.3
   insists more before the breaker cuts off (count the network requests).
2. **`BREAKER_THRESHOLD = 10`** — in 4.3, the breaker no longer opens: each call pays for all
   its attempts. That is the cost of a breaker that is too tolerant.
3. **`TIMEOUT_S = 1.0`** — in 4.4, the slow response gets through: it is up to you to say whether
   a 0.5 s wait is acceptable for your use case.
4. **`SEMANTIC_THRESHOLD = 0.5`** — in 4.6, the rephrasing is served… and a question about a
   recipe might be too: add it to the list and watch.
5. **`DATA_CLASS = "regulated"`** — all scenarios refuse before calling. Then, in
   `config/governance_policy.toml`, allow `us` for `regulated` and rerun: the policy lives
   in the configuration, not in the code.
6. **`MODEL = "openai/gpt-5.6-luna"`** — the simulated scenarios stay on the Anthropic provider
   (the script is an Anthropic script), but the costs in part 7 change.

## Part 7 — The numbers

This notebook measures **mechanisms**, not quality: there is no literature score to
compare against here (that starts in notebook 2 for embeddings, and in phase 2 for everything else).
What we can quantify right now is **what a call costs** depending on the model, and what
it becomes at scale.

In [19]:
pricing = LLMClient.from_settings().pricing
typical = Usage(input_tokens=1_000, output_tokens=300)  # a typical RAG call: context + short answer
cached = Usage(
    input_tokens=200, cache_read_tokens=800, output_tokens=300
)  # the same, prefix served by the cache

print(f"price table verified on {pricing.verified_on} — {pricing.source}\n")
print(f"{'model':<24} {'1 call':>10} {'1 call (cache)':>16} {'100k calls/month':>18} {'cache':>10}")
for model in sorted(pricing.models):
    if pricing.models[model].output_per_mtok == 0:
        continue  # embeddings: no output
    one = pricing.cost_usd(model, typical)
    with_cache = pricing.cost_usd(model, cached)
    monthly = one * 100_000
    saving = f"−{(1 - with_cache / one) * 100:.0f}%" if with_cache is not None else "n/a"
    # Unpublished cache price (Haiku 5.5): the testbench displays "unknown", never an estimate.
    cache_col = f"{with_cache:>14.5f} $" if with_cache is not None else f"{'unknown':>16}"
    print(f"{model:<24} {one:>9.5f} $ {cache_col} {monthly:>16,.0f} $ {saving:>10}")

price table verified on 2026-10-19 — Anthropic: models reference of 2026-10-06 (Claude 5.5 family added on 2026-10-19; cache: read 0.1x and write 1.25x, except Opus 5.5 whose cache read is published at 0.20); OpenAI: official pricing page recorded on 2026-09-03, unchanged

model                        1 call   1 call (cache)   100k calls/month      cache
claude-haiku-4-5           0.00250 $        0.00178 $              250 $       −29%
claude-haiku-5-5           0.00025 $          unknown               25 $        n/a
claude-opus-5              0.01250 $        0.00890 $            1,250 $       −29%
claude-opus-5-5            0.01000 $        0.00696 $            1,000 $       −30%
claude-sonnet-5            0.00500 $        0.00356 $              500 $       −29%
claude-sonnet-5-5          0.00500 $        0.00356 $              500 $       −29%
gpt-5-mini                 0.00085 $        0.00067 $               85 $       −21%
gpt-5-nano                 0.00017 $        0.00013 $  

## Interview questions

<details><summary><b>1. What is the difference between a retry and a circuit breaker? Why both?</b></summary>

The retry reasons per call: "this attempt failed, the next one may succeed" (429, 5xx,
dropped connection). The breaker reasons per backend: "it is going down, let us stop calling it". Without
a breaker, a prolonged outage makes each caller pay for all its attempts and bombards a service that is
already down; without a retry, an isolated 429 becomes a user error. The breaker only counts the
errors that say something about the backend's health (unavailable, timeout) — not 400s (our fault) nor
429s (a limit, not an outage).
</details>

<details><summary><b>2. Why jitter in the backoff?</b></summary>

After an outage, all the clients that failed at the same second would retry at the same
second: a synchronized wave that reproduces the overload (the "thundering herd"). A random
jitter of ±25% spreads the wave.
</details>

<details><summary><b>3. How do you guarantee that no sensitive data goes to the wrong provider?</b></summary>

Not by a team rule: by construction. Each request carries a mandatory data class;
each backend declares dated governance metadata; a policy
(configuration file) decides with deny-by-default, before any network call, and the decision
is traced. Embeddings and the judge go through the same policy: text leaves through those
paths too.
</details>

<details><summary><b>4. What goes into the key of an exact cache, and what makes a semantic cache dangerous?</b></summary>

Everything that influences the response (backend, model, messages, parameters, schema) and nothing
else (not the trace metadata). The semantic cache is dangerous because it serves an answer to
a *close* question, not an identical one: you need a namespace that freezes the context (system
prompt, model, parameters), a high threshold, to exclude conversations with history, and to measure
its false-positive rate.
</details>

<details><summary><b>5. Why not retry a streaming call?</b></summary>

Because tokens have already been sent to the client. Replaying the call would produce a different text
(non-determinism) and would duplicate the beginning. We retry before the first chunk; after that, we
report the error and let the caller decide. The breaker, for its part, is indeed fed by the failure.
</details>

<details><summary><b>6. How do you compute the cost of a call, and what do you do with an unknown price?</b></summary>

Normalized usage (uncached input tokens, output tokens, cache reads and writes)
multiplied by a versioned, dated price table. A model missing from the table yields an unknown
cost (`None`), traced as such; never a zero that would skew a table further down the line.
</details>

## What we did not do, and why

- **No batch API** (half price, results within 24 h): it arrives with the harness of
  phase 2, when thousands of judge calls will justify it.
- **No automatic fallback to another backend** on an outage: the fallback is a governance
  decision (which backend is allowed to receive this request?), hence the caller's, traced.
- **Per-process budget**: the counter is in memory; phase 5 will share it (Postgres) across
  API replicas.
- **In-memory semantic cache here**: the Postgres/pgvector version arrives with ingestion
  (notebook 2), on the same database.
- **No trace backend plugged in**: the spans are exported in memory to be read here;
  Langfuse receives them over OTLP (infra guide 07).

**Next**: notebook 2, from document to indexed chunk — and the testbench's first measurement (three
embedding models on SciFact, compared to the literature).

## Summary glossary

| Term | Definition |
|---|---|
| **Provider** | The remote service that runs the model (Anthropic, OpenAI, OpenRouter, Bedrock, a vLLM of our own). |
| **Backend** | An entry of `config/llm_backends.toml`: a configured provider, with its governance metadata. |
| **Token** | The billed unit of text (~4 characters in English). |
| **Completion** | The response generated by the model. |
| **TTFT** | Time to first token: delay before the first chunk in streaming. |
| **Exponential backoff** | Delay between attempts that doubles with each failure, capped. |
| **Jitter** | Random part of the delay, to desynchronize the clients. |
| **Circuit breaker** | Per-backend failure counter that cuts off calls for a while (closed / open / half-open). |
| **Retry-After** | HTTP header through which a provider indicates when to retry. |
| **Finish reason** | Why the model stopped: `stop` (done), `length` (truncated), `refusal`… |
| **Structured output** | Response constrained to a JSON schema, validated by a Pydantic model. |
| **Exact / semantic cache** | Response reused for an identical request / for a question similar in embedding terms. |
| **Embedding** | Numeric vector representing the meaning of a text; two close texts have close vectors. |
| **Data class** | Sensitivity label of a request (`public` … `regulated`), mandatory. |
| **Prompt caching** | Reuse by the provider of an identical prompt prefix, billed much cheaper. |
| **GenAI semantic conventions** | Standard OpenTelemetry attribute names for LLM calls (`gen_ai.*`). |
| **Span** | The unit of a trace: an operation, its duration, its attributes. |